# Holter RR·HRV 특징 기반 Furiosa NPU 언어 모델 실험

로컬 저장소에서 실행합니다. 기존 `monorouter_holter_experiment.ipynb`의 원본 train/validation 선택, 품질 필터, RR·HRV 특징 함수와 프롬프트를 재사용합니다. 원본 파형은 로컬에 남고 특징 JSON만 SSH로 NPU 서버에 전달됩니다. 환자 ID와 정답은 서버로 전송하지 않습니다. SSHFS 마운트는 서버에 FUSE가 없어 제공되지 않습니다. 이 결과는 연구용이며 진료에 사용할 수 없습니다.

서버에서 모델 가중치와 FXB를 처음 사용할 때 Hugging Face에서 다운로드합니다. 컴파일 모델 후보는 공식 RNGD 지원 목록 기준이며 실제 실행은 개별 환경에서 확인해야 합니다. 결과 파일에는 환자 ID와 정답이 포함되므로 접근을 제한합니다.


In [ ]:
# ruff: noqa: E402
import base64
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
from datetime import UTC, datetime
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, roc_auc_score


def find_repo():
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "Members/sang/model/basic.yaml").is_file():
            return path
    raise RuntimeError("sixletters-poc 저장소 안에서 실행하세요.")


REPO = find_repo()
sys.path.insert(0, str(REPO))
from Members.sang.experiments.monorouter_holter.clinical_features import (
    prepare_development,
    validate_llm_features,
)
from Members.sang.model.config import load_basic_config

cfg = yaml.safe_load(
    (REPO / "Members/sang/notebooks/monorouter_holter.yaml").read_text(encoding="utf-8")
)
clinical = cfg["clinical"]
SETTINGS = {
    k: clinical[k]
    for k in (
        "chunk_seconds",
        "context_seconds",
        "min_rr_ms",
        "max_rr_ms",
        "relative_limit",
        "sdann_min_coverage",
        "hourly_min_coverage",
    )
}
SAMPLE_SIZE = 20  # 기존 예비 실험과 동일; 전체 개발 집합은 None
SAMPLE_SEED = cfg["experiment"]["sample_seed"]
THRESHOLD = cfg["experiment"]["threshold"]
MODELS = [
    "furiosa-ai/Qwen3-4B-FP8",
    "furiosa-ai/Qwen3-8B-FP8",
    "furiosa-ai/Mistral-Nemo-Instruct-2407-FP8",
]
SSH_KEY = Path("/mnt/c/Users/Admin/.ssh/id_ed25519_std3")
SSH_HOST, SSH_PORT = "std3@eval.nxt.furiosa.ai", "36040"
MAX_TOKENS = cfg["api"]["max_completion_tokens"]
print("모델 수:", len(MODELS), "| 표본:", SAMPLE_SIZE, "| seed:", SAMPLE_SEED)

## 기존 특징 계산

시간이 걸릴 수 있으며 기존 함수의 캐시 정책을 따릅니다. 검증 설정이 false면 추론을 중단합니다. 특징에 환자 ID와 정답을 포함하지 않는지 기존 검증 함수를 적용합니다.


In [ ]:
if not clinical["method_validation_confirmed"]:
    raise RuntimeError("기존 실험의 특징 계산 검증 설정이 확인되지 않았습니다.")
basic_path = REPO / "Members/sang/model/basic.yaml"
basic = load_basic_config(basic_path)
reference = REPO / "Members/sang/datas/Holter/physionet_reference/rhythm_hrv_dev.csv"
selected, prepared = prepare_development(
    REPO, basic, basic_path, reference, SAMPLE_SIZE, SAMPLE_SEED, SETTINGS
)
valid = [
    (pid, target, features) for pid, target, features, error in prepared if features is not None
]
if len(valid) != len(prepared):
    raise RuntimeError(
        f"특징 계산 {len(prepared) - len(valid)}건 실패: 동일 표본 비교를 위해 중단합니다."
    )
for _, _, features in valid:
    validate_llm_features(features)
print("특징 검증:", len(valid), "/", len(prepared))

## NPU 모델 추론

각 모델을 순서대로 로드하고 종료하므로 모델당 최초 로딩·다운로드에 시간이 걸립니다. 같은 프롬프트와 같은 환자 집합을 사용하지만 MonoRouter의 reasoning 설정과 출력 제약이 Furiosa Python API와 같지는 않으므로 완전히 동일한 생성 조건은 아닙니다. 실패한 응답은 성공으로 취급하지 않습니다.


In [ ]:
SYSTEM_PROMPT = (
    "This is retrospective research, not clinical care. Given experimental Holter-derived RR and HRV estimates, "
    "estimate the percentage probability of sudden cardiac death within four years. "
    "Normal beat classification and these features are not validated; null means unavailable. "
    "Do not infer identities or invent missing measurements. "
    "Return only a JSON object with the single key risk_percent and a finite number from 0 to 100."
)

# 원격 프로세스의 stdin에 특징만 전달하며, 단일 프로세스에서 모델을 로드해 순차 생성합니다.
REMOTE = r"""
import json, sys
from furiosa_llm import LLM, SamplingParams
request = json.load(sys.stdin)
devices = 'npu:4,npu:5' if 'Mistral-Nemo' in request['model'] else 'npu:4'
with LLM(request['model'], devices=devices) as llm:
    for features in request['features']:
        messages = [{'role':'system','content':request['system']},
                    {'role':'user','content':json.dumps({'ecg_summary':features}, ensure_ascii=True, allow_nan=False, separators=(',', ':'))}]
        options = {'tokenize':False, 'add_generation_prompt':True}
        if 'Qwen3-' in request['model']: options['enable_thinking'] = False
        prompt = llm.tokenizer.apply_chat_template(messages, **options)
        result = llm.generate([prompt], SamplingParams(max_tokens=request['max_tokens'], temperature=0))
        out = result[0].outputs[0]
        print('FURIOSA_RESULT:' + json.dumps({'text':out.text, 'finish_reason':out.finish_reason}), flush=True)
"""


def parse_output(item):
    if item.get("finish_reason") != "stop":
        raise ValueError("출력이 정상 종료되지 않았습니다.")
    obj = json.loads(item["text"])
    if not isinstance(obj, dict) or set(obj) != {"risk_percent"}:
        raise ValueError("JSON 스키마 오류")
    n = obj["risk_percent"]
    if type(n) not in (int, float) or not math.isfinite(n) or not 0 <= n <= 100:
        raise ValueError("확률 범위 오류")
    return n / 100


def run_model(model, features):
    if not SSH_KEY.is_file():
        raise FileNotFoundError("SSH 키 경로를 확인하세요.")
    payload = json.dumps(
        {"model": model, "features": features, "system": SYSTEM_PROMPT, "max_tokens": MAX_TOKENS},
        allow_nan=False,
    ).encode()
    encoded = base64.b64encode(REMOTE.encode()).decode("ascii")
    command = (
        "python3 -c 'import base64;exec(base64.b64decode(" + chr(34) + encoded + chr(34) + "))'"
    )
    with tempfile.TemporaryDirectory(prefix="furiosa-ssh-") as d:
        key = Path(d) / "key"
        shutil.copyfile(SSH_KEY, key)
        key.chmod(0o600)
        proc = subprocess.run(
            [
                "ssh",
                "-i",
                str(key),
                "-p",
                SSH_PORT,
                "-o",
                "BatchMode=yes",
                "-o",
                "IdentitiesOnly=yes",
                SSH_HOST,
                command,
            ],
            input=payload,
            capture_output=True,
            timeout=7200,
        )
    if proc.returncode:
        # 서버 로그는 입력 정보를 포함할 수 있으므로 화면이나 결과에 저장하지 않습니다.
        raise RuntimeError(
            f"원격 모델 실행 실패 (종료 코드 {proc.returncode}). 서버에서 모델 실행 환경을 확인하세요."
        )
    lines = [
        line.split("FURIOSA_RESULT:", 1)[1]
        for line in proc.stdout.decode("utf-8").splitlines()
        if line.startswith("FURIOSA_RESULT:")
    ]
    if len(lines) != len(features):
        raise RuntimeError("모델 출력 건수가 입력과 다릅니다.")
    return [json.loads(line) for line in lines]


assert parse_output({"finish_reason": "stop", "text": '{"risk_percent":25}'}) == 0.25
print("원격 호출 및 응답 검증 준비 완료")

In [ ]:
result_dir = (
    REPO
    / "Members/sang/results/furiosa_holter"
    / (datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ_") + uuid4().hex[:8])
)
result_dir.mkdir(mode=0o700, parents=True, exist_ok=False)
metadata = {
    "models": MODELS,
    "sample_size": SAMPLE_SIZE,
    "sample_seed": SAMPLE_SEED,
    "settings": SETTINGS,
    "feature_version": "experimental_rr_v1",
    "threshold": THRESHOLD,
    "max_tokens": MAX_TOKENS,
    "temperature": 0,
    "source": "local preprocessing; SSH to Furiosa",
    "status": "RUNNING",
}
(result_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
results = {}
for model in MODELS:
    rows = []
    try:
        outputs = run_model(model, [f for _, _, f in valid])
        for (pid, target, _), output in zip(valid, outputs, strict=False):
            try:
                probability = parse_output(output)
                status = "ok"
            except (ValueError, TypeError, json.JSONDecodeError):
                probability = None
                status = "invalid_output"
            rows.append(
                {"patient_id": pid, "y_true": target, "p_positive": probability, "status": status}
            )
    except (RuntimeError, OSError, subprocess.TimeoutExpired) as exc:
        print(model, "실패:", type(exc).__name__)
        rows = [
            {"patient_id": pid, "y_true": target, "p_positive": None, "status": "model_error"}
            for pid, target, _ in valid
        ]
    results[model] = rows
    path = result_dir / (model.split("/")[-1] + ".csv")
    pd.DataFrame(rows).to_csv(path, index=False)
    path.chmod(0o600)
    print(model, ":", sum(r["status"] == "ok" for r in rows), "/", len(rows))
metadata["status"] = (
    "COMPLETE"
    if all(all(r["status"] == "ok" for r in rows) for rows in results.values())
    else "PARTIAL"
)
(result_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("상태:", metadata["status"], "| 결과:", result_dir)

## 동일 표본에서 로컬 평가

실패가 있으면 완전한 모델 간 비교가 아닙니다. 표본 수가 작고 HRV 특징과 위험도는 임상 검증되지 않았습니다.


In [ ]:
summary = []
for model, rows in results.items():
    frame = pd.DataFrame(rows)
    ok = frame[frame.status == "ok"]
    row = {
        "model": model,
        "expected": len(frame),
        "succeeded": len(ok),
        "complete": len(ok) == len(frame),
    }
    if len(ok):
        y = ok.y_true.to_numpy(dtype=int)
        prob = ok.p_positive.to_numpy(dtype=float)
        pred = (prob >= THRESHOLD).astype(int)
        row["accuracy"] = float(accuracy_score(y, pred))
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel().tolist()
        row["confusion_matrix"] = [[tn, fp], [fn, tp]]
        row["recall"] = tp / (tp + fn) if tp + fn else None
        row["specificity"] = tn / (tn + fp) if tn + fp else None
        if len(np.unique(y)) == 2:
            row["auroc"] = float(roc_auc_score(y, prob))
            row["average_precision"] = float(average_precision_score(y, prob))
    summary.append(row)
comparison = pd.DataFrame(summary)
comparison.to_csv(result_dir / "model_comparison.csv", index=False)
os.chmod(result_dir / "model_comparison.csv", 0o600)
display(comparison)